# Daily Practice — 2026-11-08 — ML Testing & MLOps: Data Contracts and Training/Serving Skew

**Dataset:** [Wine Quality (red)](https://archive.ics.uci.edu/dataset/186/wine+quality) — 1,599 real
physicochemical measurements of Portuguese "Vinho Verde" red wine samples (acidity, sugar, chlorides,
sulfur dioxide, density, pH, sulphates, alcohol) with an expert sensory `quality` score (3–8), loaded
from the `jbrownlee/Datasets` mirror of the classic UCI dataset.

## Problem statement

You've joined the QA/MLOps team supporting a "good wine" screening model: a binary classifier that
flags bottles likely to score `quality >= 7` (the top ~14% of this dataset) for a human sommelier's
second opinion. The data science team has a training notebook that works. Your job is to build the
automated checks that would run **before** any new model version ships — the kind of checks an SQA
engineer moving into ML would own, because "the notebook ran without an error" is not the same as "this
is safe to deploy."

Two specific failure classes you're guarding against:

1. **Bad input data reaching training or serving** — nulls, out-of-range sensor values, duplicate rows,
   wrong dtypes — anything a naive `df = pd.read_csv(...)` will happily accept without complaint.
2. **Training/serving skew** — the single most common silent killer in real ML systems. The model is
   trained on features built one way, then served in production with a *slightly* different feature
   pipeline (a forgotten scaler, a stale encoder, a copy-pasted transform that skips one column). Nothing
   crashes. The model just quietly gets much worse, usually with no immediate ground-truth labels
   available to notice from accuracy alone.

## What you should produce

1. A `validate_schema` data-contract test that checks column presence, dtypes, null counts, physically
   plausible ranges, and duplicate rows on the raw data — and a decision about what to do with any
   violations you find (this dataset has at least one; don't pre-clean it away before running the check).
2. A training pipeline: dedup → train/test split → a fitted `StandardScaler` → a `preprocess` function,
   plus a `test_preprocessing_determinism` check that calling `preprocess` twice on identical input gives
   identical output (a basic reproducibility guarantee).
3. A baseline `LogisticRegression` model with accuracy **and** precision/recall/F1 reported for the
   positive ("good wine") class specifically — because this is an imbalanced label, and accuracy alone
   would hide how the model actually performs on the class that matters.
4. A deliberately *buggy* `buggy_serving_preprocess` function simulating a realistic production bug (you
   choose the bug — e.g. a feature left unscaled, or scaled with different stats), and a
   `detect_training_serving_skew` test that compares per-feature statistics between the training-time
   features and the "serving" features, flagging anything that drifted beyond a threshold.
5. A short written verdict (3–5 sentences): would an accuracy-only monitoring dashboard have caught this
   bug before real users saw degraded predictions, and why does checking feature statistics at serving
   time catch it earlier than waiting for ground-truth labels?

Try it yourself in the starter cells below before you scroll down to the solution.

## Setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Load the data

The raw CSV has no header row and is comma-separated in this mirror (the original UCI file is
semicolon-separated — another thing that's bitten real pipelines). We assign the standard UCI Wine
Quality column names ourselves.

In [ ]:
WINE_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/winequality-red.csv"

COLUMNS = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides",
    "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH", "sulphates",
    "alcohol", "quality",
]
FEATURE_COLUMNS = COLUMNS[:-1]

raw_df = pd.read_csv(WINE_URL, header=None, names=COLUMNS)
raw_df.shape

## Your task

Fill in the TODOs below. Signatures are given — don't change them, the solution cells assume these
names.

In [ ]:
# A physically-plausible range for each column, used by validate_schema below.
# (Loose enough to accept genuine extreme wines, tight enough to catch garbage like a negative pH
# or a sensor fault reading 9999 for chlorides.)
SCHEMA = {
    "fixed_acidity": (0, 20),
    "volatile_acidity": (0, 2),
    "citric_acid": (0, 1.2),
    "residual_sugar": (0, 20),
    "chlorides": (0, 1),
    "free_sulfur_dioxide": (0, 100),
    "total_sulfur_dioxide": (0, 320),
    "density": (0.98, 1.01),
    "pH": (2.5, 4.5),
    "sulphates": (0, 2.5),
    "alcohol": (7, 16),
    "quality": (0, 10),
}


def validate_schema(df, schema=SCHEMA):
    """TODO: return a list of human-readable violation strings (empty list = pass). Check:
    - every column in `schema` is present in `df`
    - every present column is numeric
    - no nulls in any schema column
    - no values outside the column's (lo, hi) range in `schema`
    - no fully-duplicated rows in `df`
    """
    violations = []
    # YOUR CODE HERE
    return violations

In [ ]:
# TODO: call validate_schema(raw_df) and print the violations.
# Do NOT drop duplicates before this call — the point is to catch them here, as a CI gate would.
violations = None

In [ ]:
def clean_and_split(df, feature_columns=FEATURE_COLUMNS, test_size=0.25, random_state=RANDOM_STATE):
    """TODO:
    1. Drop fully-duplicated rows from `df`.
    2. Build the binary label y = 1 if quality >= 7 else 0.
    3. Split into X_train, X_test, y_train, y_test using train_test_split with the given
       test_size and random_state, and stratify=y (this label is imbalanced — don't skip stratify).
    Return X_train, X_test, y_train, y_test as pandas objects (don't convert to numpy yet).
    """
    # YOUR CODE HERE
    return None, None, None, None


X_train, X_test, y_train, y_test = clean_and_split(raw_df)

In [ ]:
def fit_scaler(X_train):
    """TODO: fit and return a sklearn StandardScaler on X_train."""
    # YOUR CODE HERE
    return None


def preprocess(X_raw, fitted_scaler):
    """TODO: apply fitted_scaler.transform to X_raw and return the result (a numpy array)."""
    # YOUR CODE HERE
    return None


scaler = fit_scaler(X_train)
X_train_scaled = preprocess(X_train, scaler)
X_test_scaled = preprocess(X_test, scaler)

In [ ]:
def test_preprocessing_determinism(X_raw, fitted_scaler):
    """TODO: call preprocess(X_raw, fitted_scaler) twice and return True if the two results are
    exactly equal (np.array_equal), False otherwise. This is the kind of check that catches a
    preprocessing step with hidden randomness or external mutable state.
    """
    # YOUR CODE HERE
    return None


test_preprocessing_determinism(X_test, scaler)

In [ ]:
def fit_and_evaluate(X_train_scaled, y_train, X_test_scaled, y_test):
    """TODO:
    1. Fit a sklearn LogisticRegression(max_iter=1000, random_state=RANDOM_STATE) on the scaled
       training data.
    2. Predict on X_test_scaled.
    3. Return (model, metrics) where metrics is a dict with keys 'accuracy', 'precision', 'recall',
       'f1' — precision/recall/F1 computed for the positive class (good wine, label 1) specifically,
       not averaged across classes.
    """
    # YOUR CODE HERE
    return None, None


model, baseline_metrics = fit_and_evaluate(X_train_scaled, y_train, X_test_scaled, y_test)
baseline_metrics

### Now simulate the production bug

Pick ONE realistic training/serving skew bug and implement it in `buggy_serving_preprocess` below —
for example, a feature that's accidentally left on its raw (unscaled) scale, or scaled with the wrong
statistics (e.g. a stale scaler fit on old data). Don't make it a crash — it should run fine and return
plausible-looking numbers, because that's what makes this bug class dangerous in real systems.

In [ ]:
def buggy_serving_preprocess(X_raw, fitted_scaler):
    """TODO: implement a realistic training/serving skew bug. Must still return an array/DataFrame
    of the same shape as preprocess() would, so the model can score it without erroring.
    """
    # YOUR CODE HERE
    return None


X_test_serving = buggy_serving_preprocess(X_test, scaler)

In [ ]:
def detect_training_serving_skew(X_train_scaled, X_serving, feature_names, threshold=0.25):
    """TODO: compare, for each feature, the training-set mean (on the SCALED training features,
    which should be ~0 for every column since they came out of a StandardScaler) against the mean of
    that same feature in X_serving. Return a pandas DataFrame with columns
    ['feature', 'train_mean', 'serving_mean', 'abs_diff', 'flagged'] where flagged = abs_diff > threshold.
    """
    # YOUR CODE HERE
    return None


skew_report = detect_training_serving_skew(X_train_scaled, X_test_serving, FEATURE_COLUMNS)
skew_report

In [ ]:
# TODO: using the buggy serving features, call model.predict and compute accuracy against y_test.
# Compare this to baseline_metrics['accuracy'] to see the real-world damage the skew test above
# would have caught *before* a single production label came back.
serving_accuracy = None
serving_accuracy

### Write-up

TODO: in 3–5 sentences — would an accuracy-only monitoring dashboard have caught this bug before real
users saw degraded predictions, given that ground-truth labels for scored wine batches typically arrive
late (if ever)? Why does comparing feature statistics at serving time catch it earlier?

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def validate_schema(df, schema=SCHEMA):
    violations = []
    for col in schema:
        if col not in df.columns:
            violations.append(f"missing column: {col}")
    for col, (lo, hi) in schema.items():
        if col not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            violations.append(f"{col}: not numeric")
            continue
        n_null = df[col].isna().sum()
        if n_null:
            violations.append(f"{col}: contains {n_null} nulls")
        out_of_range = ((df[col] < lo) | (df[col] > hi)).sum()
        if out_of_range:
            violations.append(f"{col}: {out_of_range} values outside [{lo}, {hi}]")
    n_dupes = df.duplicated().sum()
    if n_dupes:
        violations.append(f"{n_dupes} duplicate rows")
    return violations

violations = validate_schema(raw_df)
print(violations)
# -> ['240 duplicate rows']
# Every column passes dtype/range checks (it's clean, real sensor data), but ~15% of rows are exact
# duplicates. That's a genuine data-quality finding this UCI file is known to have, and a schema test
# is exactly where an SQA engineer would catch it before it silently inflates the training set and
# leaks near-duplicate rows across train/test.


def clean_and_split(df, feature_columns=FEATURE_COLUMNS, test_size=0.25, random_state=RANDOM_STATE):
    clean = df.drop_duplicates().reset_index(drop=True)
    y = (clean["quality"] >= 7).astype(int)
    X = clean[feature_columns]
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

X_train, X_test, y_train, y_test = clean_and_split(raw_df)


def fit_scaler(X_train):
    return StandardScaler().fit(X_train)

def preprocess(X_raw, fitted_scaler):
    return fitted_scaler.transform(X_raw)

scaler = fit_scaler(X_train)
X_train_scaled = preprocess(X_train, scaler)
X_test_scaled = preprocess(X_test, scaler)


def test_preprocessing_determinism(X_raw, fitted_scaler):
    a = preprocess(X_raw, fitted_scaler)
    b = preprocess(X_raw, fitted_scaler)
    return np.array_equal(a, b)

test_preprocessing_determinism(X_test, scaler)
# -> True


def fit_and_evaluate(X_train_scaled, y_train, X_test_scaled, y_test):
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    metrics = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    }
    return model, metrics

model, baseline_metrics = fit_and_evaluate(X_train_scaled, y_train, X_test_scaled, y_test)
baseline_metrics
# -> {'accuracy': 0.871, 'precision': 0.538, 'recall': 0.304, 'f1': 0.389}
# Accuracy alone (87%) looks great. Recall on the class that matters (catching actually-good wine) is
# only 30% -- the model misses 7 out of 10 good wines. This is the "evaluation metric failure mode"
# half of the lesson: on an imbalanced label, accuracy is the wrong headline number.


def buggy_serving_preprocess(X_raw, fitted_scaler):
    # Realistic bug: someone refactored the serving pipeline and a late-added column (`alcohol`)
    # got attached AFTER the scaling step instead of before it, so it reaches the model on its raw
    # scale (8-15) instead of standardized (~N(0,1)). Everything still runs -- no exception anywhere.
    scaled = fitted_scaler.transform(X_raw)
    scaled_df = pd.DataFrame(scaled, columns=X_raw.columns, index=X_raw.index)
    scaled_df["alcohol"] = X_raw["alcohol"].values
    return scaled_df.values

X_test_serving = buggy_serving_preprocess(X_test, scaler)


def detect_training_serving_skew(X_train_scaled, X_serving, feature_names, threshold=0.25):
    train_means = np.asarray(X_train_scaled).mean(axis=0)
    serving_means = np.asarray(X_serving).mean(axis=0)
    rows = []
    for i, name in enumerate(feature_names):
        diff = abs(serving_means[i] - train_means[i])
        rows.append({
            "feature": name,
            "train_mean": train_means[i],
            "serving_mean": serving_means[i],
            "abs_diff": diff,
            "flagged": diff > threshold,
        })
    return pd.DataFrame(rows)

skew_report = detect_training_serving_skew(X_train_scaled, X_test_serving, FEATURE_COLUMNS)
skew_report[skew_report["flagged"]]
# ->   feature  train_mean  serving_mean  abs_diff  flagged
#      alcohol   ~0.0        10.50         10.50     True
# The skew test catches exactly the one feature the "refactor" broke, with no ground-truth labels
# involved at all -- it only needs the serving batch's own feature values.


pred_serving = model.predict(X_test_serving)
serving_accuracy = accuracy_score(y_test, pred_serving)
serving_accuracy
# -> 0.135
# Real-world damage: accuracy collapses from 0.871 to 0.135 (worse than always predicting the
# majority class). A dashboard watching accuracy would eventually show this -- but only once enough
# production labels trickle back in, which for a "second opinion from a human sommelier" workflow
# could be days or weeks. The skew test flags it at serving time, on the very first batch, using only
# the inputs the model is about to score.
```

**Write-up answer:** No — an accuracy-only dashboard is a lagging indicator: it needs ground-truth
labels, which for this product arrive late (a sommelier has to actually taste the flagged bottles).
During that gap, every wine scored by the model is scored on the broken pipeline. A training/serving
skew test needs no labels at all; it only compares the statistical shape of the serving-time features
against what the model was trained on, so it can fire on the very first production batch — turning a
days-or-weeks-long silent failure into an immediate, label-free CI/monitoring gate. This is the general
argument for treating "does the served feature distribution look like the training distribution" as a
release gate, not just final-metric accuracy.

</details>